# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to access, explore, and analyze the FAIR<sup>2</sup> dataset using the `mlcroissant` library. All references to record sets, fields, and columns will be made through their `@id` identifiers in line with best practices.

## Dataset Source
- Croissant schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
- License: [Open Data Commons By 1.0](https://opendatacommons.org/licenses/by/1-0/)
- Data papers and more metadata are accessible via [SEN SCIENCE](https://sen.science/doi/10.71728/senscience.qs2f-h81p).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and connect to its record sets using the `mlcroissant` library. This will give us an overview of the dataset and access to further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'Unknown')}")

## 2. Data Overview

Let's summarize the record sets available in the dataset, along with their `@id`s, fields, and columns. This step is critical for navigating the dataset programmatically using `mlcroissant`.

In [ ]:
# List available record sets and their fields by @id
# All references will use @id per best practice
record_sets = dataset.record_sets

print("Available RecordSets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    name = rs.get('name', rs.get('rdfs:label', 'Unnamed'))
    print(f"  name: {name}")
    print(f"  description: {rs.get('description', '')}")
    # List fields in this record set
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        print(f"  Fields:")
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"    - {field_id}")
    print()

# Pick the first available record set for preview
if len(record_sets) > 0:
    first_rs_id = record_sets[0]['@id']
    print(f"\nPreviewing a few records from record set {first_rs_id}:")
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        if i>=2: break
        print(rec)

## 3. Data Extraction

We next load data from one or more record sets into pandas DataFrames for further analysis and exploration. Each DataFrame is labeled by the record set `@id`. Use the `@id` as the key throughout to maintain direct schema mapping.

In [ ]:
# Extract data from all available record sets
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    # Each yields a generator of records (as dicts with field @id as keys)
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from '{rs_id}'")
    else:
        print(f"No records for record set '{rs_id}'")

# Show columns for the main record set (fallback to first one if unsure)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes and not dataframes[main_record_set_id].empty:
    print(f"\nColumns in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No data to display. Review record sets above.")

## 4. Exploratory Data Analysis (EDA)

Let's explore some numeric and categorical fields in the dataset. Example analysis: filtering patients by age, normalizing, and summarizing by sex. All field references use their `@id`.

In [ ]:
# EDA on the main record set (replace field ids below as appropriate for your dataset)
df = dataframes.get(main_record_set_id)
if df is not None and not df.empty:
    # Try to auto-detect a likely numeric (age) and group (sex) field by @id
    import re
    field_ids = df.columns.tolist()
    age_field_id = next((fid for fid in field_ids if re.search(r'age', fid, re.IGNORECASE)), None)
    sex_field_id = next((fid for fid in field_ids if re.search(r'sex|gender', fid, re.IGNORECASE)), None)

    print(f"Auto-selected age numeric field: '{age_field_id}'")
    print(f"Auto-selected grouping field: '{sex_field_id}'")

    # Only proceed if found
    if age_field_id and sex_field_id:
        # Filter for age > 50 as a clinical threshold
        try:
            df[age_field_id] = pd.to_numeric(df[age_field_id], errors='coerce')
            filtered_df = df[df[age_field_id] > 50].copy()
            print(f"Filtered records with {age_field_id} > 50: {len(filtered_df)} results\n")
            # Normalize age
            filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
            print(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

            # Group by sex and show mean age
            grouped_df = filtered_df.groupby(sex_field_id)[age_field_id].mean().reset_index()
            print(f"\nMean {age_field_id} by {sex_field_id}:")
            print(grouped_df)
        except Exception as e:
            print(f"Could not process numeric/group field: {e}")
    else:
        print("Suitable numeric (age) or group (sex) field not detected. Please check the schema field @id list above.")
else:
    print("No main record set DataFrame available for EDA.")

## 5. Visualization

Let's visualize the distribution of age and the age breakdown by sex (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and age_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[age_field_id].dropna(), bins=10, kde=True, color='skyblue')
    plt.xlabel('Patient Age')
    plt.title('Age Distribution')
    plt.show()

    if sex_field_id in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=df[sex_field_id], y=df[age_field_id])
        plt.xlabel(sex_field_id)
        plt.ylabel('Age')
        plt.title(f'Age Distribution by {sex_field_id}')
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load and process the FAIR<sup>2</sup> colorectal cancer survivors dataset using the `mlcroissant` library, referencing all entities strictly by their schema `@id`. We've:
- Loaded and described the available record sets
- Explored field structure and previewed records
- Filtered and normalized patient age data, and grouped by sex
- Visualized age distribution overall and by sex

For further analysis, repeat the pattern above and reference the record set/field/column with their respective `@id`s. You may access more fields or perform clinical/statistical studies as required, always referring to the Croissant schema for interoperability.